**Run evidence** · task `08a_create_jobs` · run `1058123584373511` · result **SUCCESS** · executed 2026-10-19 17:25 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `08_jobs/08a_create_jobs`.

# 08a — Create the Lakeflow Job: raw → gold → quality gate → semantics → Lakebase

Registers one **Lakeflow Job** (serverless) that refreshes everything the business
sees, as a graph of tasks instead of one big notebook:

```
simulate_raw_sap
  ├─ gold_material ─┬─ gold_sourcing ─────────┐
  └─ gold_supplier ─┼─ gold_supplier_risk* ───┼─ constraints_comments ─ data_quality_gate ─ dq_passed?
                    └─ gold_purchase_order ───┘                                             │ true
                    ┌──────────────────────────────┬────────────────────────────────────────┤
             sql_functions                  metric_views                          value_dictionaries
                    └──────────────────────────────┴──────── sync_lakebase_worklist ────────┘
```

`*` = a **Lakeflow Declarative Pipeline** task: Auto Loader ingests the SAP Ariba
Supplier Risk export files into `RAW_SUPPLIER_RISK` and builds the gold
`SUPPLIER_RISK` materialized view (`02_build_gold/02d_supplier_risk_pipeline.sql`).
Every other task is a serverless notebook.

What it shows:
* **Job parameters** (`catalog`, `schema`, `warehouse_id`) flow into every task.
* **Parallel fan-out / fan-in** along the real dependencies of the gold build.
* A **data-quality gate** (`08b`) plus a **condition task**: if a check fails, the
  semantic layer and Lakebase are not refreshed (downstream tasks are skipped).
* The last task lands the results where the business acts: the **metric views** and
  the **Lakebase** risk worklist (`10b`).
* A daily **schedule** (created paused) and serverless compute (no cluster config).

Idempotent: a job with the same name is reset, not duplicated. The earlier
single-task job `sap_purchasing_data_prep` is removed if present.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()

# Run the job once after creating it and wait for the result (validates the graph).
RUN_NOW = True

ROOT = demo_root()
print(f"Demo folder: {ROOT}")

# The Lakeflow Declarative Pipeline for the Ariba slice (created/updated, not run here).
PIPELINE_ID = ensure_supplier_risk_pipeline()

Demo folder: /Workspace/Users/max.kohler@databricks.com/sap-purchasing-genie-demo
✓ Updated pipeline 'sap_purchasing_ariba_supplier_risk_purchasing_demo' (1146b932-5e41-4e0e-8c98-5a73903c2dac)


## Define the task graph

In [ ]:

def nb(key, path, depends_on=(), outcome=None, timeout=3600):
    """A serverless notebook task (no compute block) reading the job parameters."""
    deps = [jobs.TaskDependency(task_key=d, outcome=outcome) for d in depends_on]
    return jobs.Task(task_key=key, depends_on=deps or None, timeout_seconds=timeout,
                     notebook_task=jobs.NotebookTask(notebook_path=f"{ROOT}/{path}", source=jobs.Source.WORKSPACE))


tasks = [
    nb("simulate_raw_sap", "_helper/01_simulate_raw_tables"),
    nb("gold_material", "02_build_gold/02a_material", ["simulate_raw_sap"]),
    nb("gold_supplier", "02_build_gold/02b_supplier", ["simulate_raw_sap"]),
    nb("gold_sourcing", "02_build_gold/02c_sourcing", ["gold_material", "gold_supplier"]),
    # Workflow D runs as a pipeline task (Auto Loader → streaming table → materialized view).
    jobs.Task(task_key="gold_supplier_risk", timeout_seconds=3600,
              depends_on=[jobs.TaskDependency(task_key="gold_supplier")],
              pipeline_task=jobs.PipelineTask(pipeline_id=PIPELINE_ID, full_refresh=False)),
    nb("gold_purchase_order", "02_build_gold/02e_purchase_order", ["gold_material", "gold_supplier"]),
    nb("constraints_comments", "02_build_gold/02f_constraints_comments",
       ["gold_sourcing", "gold_supplier_risk", "gold_purchase_order"]),
    nb("data_quality_gate", "08_jobs/08b_data_quality_gate", ["constraints_comments"]),
    jobs.Task(task_key="dq_passed", depends_on=[jobs.TaskDependency(task_key="data_quality_gate")],
              condition_task=jobs.ConditionTask(op=jobs.ConditionTaskOp.EQUAL_TO,
                                                left="{{tasks.data_quality_gate.values.dq_passed}}",
                                                right="true")),
    nb("sql_functions", "03_semantics/03a_sql_functions", ["dq_passed"], outcome="true"),
    nb("metric_views", "03_semantics/03b_metric_views", ["dq_passed"], outcome="true"),
    nb("value_dictionaries", "03_semantics/03c_value_dictionaries", ["dq_passed"], outcome="true"),
    nb("sync_lakebase_worklist", "10_lakebase/10b_sync_worklist",
       ["sql_functions", "metric_views", "value_dictionaries"]),
]

settings = dict(
    name=CFG.job_name,
    description=("SAP purchasing demo pipeline: simulate SAP extracts, ingest the Ariba supplier-risk "
                 "files with a Lakeflow Declarative Pipeline, build the governed gold layer, "
                 "gate on data quality, refresh SQL functions + metric views, and sync the sourcing-risk "
                 "worklist into Lakebase for the App."),
    tasks=tasks,
    parameters=[jobs.JobParameterDefinition(name="catalog", default=CFG.catalog),
                jobs.JobParameterDefinition(name="schema", default=CFG.schema),
                jobs.JobParameterDefinition(name="warehouse_id", default=CFG.warehouse_id)],
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 6 * * ?", timezone_id="Europe/Berlin",
                               pause_status=jobs.PauseStatus.PAUSED),
    queue=jobs.QueueSettings(enabled=True),
    max_concurrent_runs=1,
    tags={"demo": "sap-purchasing-genie"},
)

## Create or update the job (idempotent by name)

In [ ]:
existing = list(w.jobs.list(name=CFG.job_name))
if existing:
    job_id = existing[0].job_id
    w.jobs.reset(job_id=job_id, new_settings=jobs.JobSettings(**settings))
    print(f"✓ Updated job '{CFG.job_name}' (job_id={job_id})")
else:
    job_id = w.jobs.create(**settings).job_id
    print(f"✓ Created job '{CFG.job_name}' (job_id={job_id})")

# The earlier single-task job wrapped 00_run_all; the Lakeflow job replaces it.
for old in w.jobs.list(name=CFG.legacy_job_name):
    w.jobs.delete(job_id=old.job_id)
    print(f"✓ Removed legacy job '{CFG.legacy_job_name}' (job_id={old.job_id})")

print(f"Open it in the UI: {w.config.host}/jobs/{job_id}")

✓ Updated job 'sap_purchasing_lakeflow_pipeline' (job_id=1038223178130579)
Open it in the UI: https://fe-sandbox-serverless-sandbox-gs5ou3.cloud.databricks.com/jobs/1038223178130579


## Run it once and wait
Gives the demo a completed run to show (task graph, durations, the DQ branch).

In [ ]:
result = "not run"
if RUN_NOW:
    run = w.jobs.run_now_and_wait(job_id=job_id, job_parameters={
        "catalog": CFG.catalog, "schema": CFG.schema, "warehouse_id": CFG.warehouse_id})
    states = {t.task_key: str(t.state.result_state).split(".")[-1] for t in run.tasks}
    for k, v in states.items():
        print(f"  {k:<24} {v}")
    result = str(run.state.result_state).split(".")[-1]
    print(f"Run {run.run_id}: {result}  → {run.run_page_url}")
    if result != "SUCCESS":
        raise RuntimeError(f"Lakeflow job run ended with {result}: {states}")

  simulate_raw_sap         SUCCESS
  gold_material            SUCCESS
  gold_supplier            SUCCESS
  gold_sourcing            SUCCESS
  gold_supplier_risk       SUCCESS
  gold_purchase_order      SUCCESS
  constraints_comments     SUCCESS
  data_quality_gate        SUCCESS
  dq_passed                SUCCESS
  sql_functions            SUCCESS
  metric_views             SUCCESS
  value_dictionaries       SUCCESS
  sync_lakebase_worklist   SUCCESS
Run 837085352084892: SUCCESS  → https://fe-sandbox-serverless-sandbox-gs5ou3.cloud.databricks.com/?o=7474656116909842#job/1038223178130579/run/837085352084892


In [ ]:
dbutils.notebook.exit(f"08a_create_jobs: SUCCESS — job_id={job_id}, run={result}")

Notebook exited: 08a_create_jobs: SUCCESS — job_id=1038223178130579, run=SUCCESS
